# 9.1 Information Theory

**Goal:** compute information measures and interpret their roles in probabilistic learning.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Information Theory](1_Information-Theory.ipynb)
2. [Statistical Learning Theory](2_Statistical-Learning-Theory.ipynb)
3. [Advanced Probability](3_Advanced-Probability.ipynb)
4. [Time-Series Statistics](4_Time-Series-Statistics.ipynb)
5. [Statistical Computing](5_Statistical-Computing.ipynb)
6. [Specialized Statistical Methods](6_Specialized-Statistical-Methods.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Entropy, Joint Entropy, and Conditional Entropy

For a discrete distribution p, entropy is $H(X)=-\sum_xp(x)\log p(x)$, with 0 log 0 interpreted as zero. Base-2 logs give bits; natural logs give nats. Entropy measures uncertainty under a distribution, not the semantic importance or correctness of outcomes. A deterministic variable has zero entropy; a fair binary variable has one bit.

**Joint entropy** uses p(x,y); **conditional entropy** is $H(Y|X)=\sum_xp(x)H(Y|X=x)=H(X,Y)-H(X)$. The chain rule is H(X,Y)=H(X)+H(Y|X). Differential entropy for continuous densities is a different quantity: it can be negative and changes with units, so discrete intuition must not be transferred blindly.

In [2]:
joint=np.array([[.4,.1],[.1,.4]])
def entropy(p):
    p=np.asarray(p);positive=p[p>0]
    return -np.sum(positive*np.log2(positive))
hx=entropy(joint.sum(1));hy=entropy(joint.sum(0));hxy=entropy(joint)
print("H(X), H(Y), H(X,Y), H(Y|X):",hx,hy,hxy,hxy-hx)

H(X), H(Y), H(X,Y), H(Y|X): 1.0 1.0 1.721928094887362 0.7219280948873621


## 2. Cross-Entropy and Kullback–Leibler Divergence

Cross-entropy $H(p,q)=-\sum p\log q$ is expected code/log-loss cost when p generates outcomes but q is predicted. KL divergence is $D_{KL}(p\|q)=\sum p\log(p/q)=H(p,q)-H(p)$. It is nonnegative when defined, zero iff distributions agree almost everywhere, asymmetric, and not a metric. If p assigns positive mass where q is zero, KL is infinite.

**Worked example:** a Bernoulli truth p=.8 scored with q=.6 has greater expected log loss than q=.8. A finite observed dataset's log loss is a noisy estimate and cannot establish the true generating p. Clipping q changes the evaluated prediction; do not confuse numerical stability with calibration.

In [3]:
p=np.array([.2,.8]);q=np.array([.4,.6])
cross=-np.sum(p*np.log2(q));kl=np.sum(p*np.log2(p/q))
print("Cross-entropy bits:",cross,"entropy:",entropy(p),"KL bits:",kl)
assert np.isclose(cross-entropy(p),kl)
assert kl>=0

Cross-entropy bits: 0.8539580943104375 entropy: 0.7219280948873623 KL bits: 0.1320299994230752


## 3. Jensen–Shannon Divergence

With equal weights and m=(p+q)/2, JS(p,q)=½KL(p||m)+½KL(q||m). It is symmetric and bounded between 0 and log 2 (one bit with base-2 logs). Its square root is a metric, while JS itself need not satisfy the metric triangle inequality. It remains finite for disjoint finite supports because the mixture covers both.

Different weighting conventions change the bound and definition. Empirical histograms must use aligned categories/bins and coherent normalization. A small JS between marginal feature distributions does not prove stability of labels or joint feature relationships.

In [4]:
m=(p+q)/2
js=.5*np.sum(p*np.log2(p/m))+.5*np.sum(q*np.log2(q/m))
print("JS bits:",js,"sqrt JS distance:",np.sqrt(js))
assert 0<=js<=1

JS bits: 0.03485155455967723 sqrt JS distance: 0.18668571064673703


## 4. Mutual Information and Information Gain

$ I(X;Y)=D_{KL}(p(x,y)\|p(x)p(y))=H(Y)-H(Y|X)$, measuring dependence rather than only linear association. It is zero iff independence for the population distribution. Conditional MI averages residual dependence given another variable; marginal MI can hide conditional patterns or selection effects.

**Information gain** for a partition/split is the parent target entropy minus the weighted child entropies; in a discrete split this equals MI with the split variable. High-cardinality splits can have large empirical gain from overfitting, so decision trees use regularization and independent validation. MI does not establish causality and plug-in estimates can have substantial finite-sample bias.

In [5]:
px=joint.sum(1);py=joint.sum(0)
mi=np.sum(joint*np.log2(joint/np.outer(px,py)))
print("MI bits:",mi,"entropy reduction:",hy-(hxy-hx))
assert np.isclose(mi,hy-(hxy-hx))
# A deterministic binary split removes all target uncertainty in this idealized table.
print("Perfect binary split information gain:",entropy([.5,.5]))

MI bits: 0.27807190511263774 entropy reduction: 0.2780719051126379
Perfect binary split information gain: 1.0


## AI/ML application and interpretation

Entropy and cross-entropy underpin classification losses, KL appears in variational objectives, and MI supports feature analysis. Keep discrete probabilities distinct from densities, use aligned supports, and account for estimator bias. Choosing a divergence shapes which mismatches the learning objective penalizes.

## Practice

1. What changes when logs use base e rather than 2?
2. Is KL symmetric or bounded by one bit?
3. Why can zero predicted probability be dangerous under log loss?
4. Does MI identify a causal direction?

## Solutions

1. Units change from bits to nats by a constant factor; distributions and substantive ordering remain the same.
2. Neither in general. Equal-weight JS has the one-bit bound, not KL.
3. A positive-probability true event assigned zero yields infinite expected log loss/KL.
4. No. MI is symmetric dependence and cannot distinguish causal direction or confounding.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](2_Statistical-Learning-Theory.ipynb).